# データ準備
## 前提
- プロジェクトルートで uv sync（取得はネット必須）
‐ raw / external は一度書いたら上書きしない（再取得は別ファイル名）
- ファイル名に 取得日 YYYYMMDD を入れる（data-catalog）
- pybaseball は pandas を返す → 保存・分析は polars に載せ替える

In [ ]:
# 共通セットアップ
from datetime import date

import polars as pl
from pybaseball import chadwick_register, pitching_stats, playerid_lookup, statcast

from analysis_project.paths import data_dir, ensure_parent_dir

FETCH_DATE = date.today().strftime("%Y%m%d")  # 例: 20260921


# 山本由伸の ID 固定（マイルストーン1）
## Keys
- key_mlbam=808967
- key_fangraphs=33825

In [ ]:
# 山本由伸の ID をそれぞれ取得する
ids_yoshi_yamamoto = playerid_lookup("yamamoto", "yoshinobu")
print(ids_yoshi_yamamoto)


# 山本由伸の ID 固定（マイルストーン2）
## 例（2026-03 時点の lookup 結果）:
## key_mlbam=808967, key_fangraphs=33825

YAMAMOTO_MLBAM = 808967
YAMAMOTO_FANG = 33825

# ID データ取得
## 保存するディレクトリを作成
register_dir = data_dir() / "external" / "register"

## 山本由伸の ID 一覧を parquet で保存する
output_path = ensure_parent_dir(register_dir / f"{FETCH_DATE}_yamamoto_ids.parquet")
pl.from_pandas(ids_yoshi_yamamoto).write_parquet(output_path)

# 全選手の ID Register を取得し、保存する（初回のみ　結合用）
reg_all_players_path = ensure_parent_dir(register_dir / f"{FETCH_DATE}_chadwick_register.parquet")

if not reg_all_players_path.exists(): # 初回のみ実行される
    reg_all_players = chadwick_register()
    pl.from_pandas(reg_all_players).write_parquet(reg_all_players_path)


  name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
0  yamamoto  yoshinobu     808967  yamay001  yamamyo01          33825   

   mlb_played_first  mlb_played_last  
0            2024.0           2026.0  


# データ取得
- Statcast:投球データ
- FanGraphs：投手成績
- MLB S多tsAPI：試合記録

In [13]:
# 投球データの取得関数
def fetch_statcast(start_dt:str, end_dt:str, label:str) -> pl.DataFrame:
    """label 例："2025_regular", "2025_post"

    Args:
        start_dt (str): _description_
        end_dt (str): _description_
        label (str): _description_

    Returns:
        pl.DataFrame: _description_
    """
    # 投球データを取得する
    ## 投球データを statcast で取得する pdf -> pandas+dataframe の意味
    pdf = statcast(start_dt=start_dt, end_dt=end_dt) # 全投球・1球1行
    df = pl.from_pandas(pdf)
    path = ensure_parent_dir(
        data_dir() / "external" / "statcast" / f"{FETCH_DATE}_{label}.parquet"
    )

    df.write_parquet(path)
    return df

In [ ]:
# 投手成績の取得関数
def fetch_fg_pitching_2025() -> pl.DataFrame:
    """2025 年の投手成績を取得する"""
    pdf = pitching_stats(2025, qual=None, split_seasons=True)
    df = pl.from_pandas(pdf)
    path = ensure_parent_dir(
        data_dir() / "raw" / "fangraphs" / f"{FETCH_DATE}_pitching_2025.parquet"
    )
    